In [4]:
import httpx
import pathlib
from decouple import config

In [5]:
NBS_DIR = pathlib.Path().resolve()
REPO_DIR = NBS_DIR.parent
DATA_DIR = REPO_DIR / "data"
GENERATED_DIR = DATA_DIR / "generated"
GENERATED_DIR.mkdir(exist_ok=True, parents=True)



In [6]:
API_ACCESS_KEY = config('API_ACCESS_KEY')

headers = {
    "X-API-Key": API_ACCESS_KEY
}
endpoint = "http://127.0.0.1:8000/predictions"
preds_res = httpx.get(endpoint, 
                params={"status": "succeeded"},
                headers=headers)
preds_json = preds_res.json()
preds_json



[{'id': 'cbd2b8b22srmt0cwv53s83n08m',
  'url': '/predictions/cbd2b8b22srmt0cwv53s83n08m',
  'status': 'succeeded',
  'created_at': '2026-03-10T19:44:19.990000Z',
  'started_at': '2026-03-10T19:44:36.069157Z',
  'completed_at': '2026-03-10T19:44:36.069157Z'},
 {'id': 'bqgkg0mhwnrmt0cwv53a0kk138',
  'url': '/predictions/bqgkg0mhwnrmt0cwv53a0kk138',
  'status': 'succeeded',
  'created_at': '2026-03-10T19:43:26.693000Z',
  'started_at': '2026-03-10T19:43:43.122835Z',
  'completed_at': '2026-03-10T19:43:43.122835Z'},
 {'id': '36v643rz6drmt0cwv3vs5fd4s8',
  'url': '/predictions/36v643rz6drmt0cwv3vs5fd4s8',
  'status': 'succeeded',
  'created_at': '2026-03-10T18:16:39.987000Z',
  'started_at': '2026-03-10T18:16:59.362632Z',
  'completed_at': '2026-03-10T18:16:59.362632Z'},
 {'id': 't1fja6t90xrmt0cwv3jspg4fym',
  'url': '/predictions/t1fja6t90xrmt0cwv3jspg4fym',
  'status': 'succeeded',
  'created_at': '2026-03-10T17:57:11.047000Z',
  'started_at': '2026-03-10T17:57:26.624489Z',
  'completed_a

In [7]:
BASE_URL="http://127.0.0.1:8000"
for pred in preds_json:
    path = pred.get('url')
    endpoint = f"{BASE_URL}{path}"
    res = httpx.get(endpoint, headers=headers)
    if res.status_code not in range(200, 299):
        continue
    data = res.json()
    files = data.get('files') or None
    if files is None:
        continue
    print(data)
    obj_id = data.get('id')
    with httpx.Client() as client:
        for i, file_path in enumerate(files):
            fname = pathlib.Path(file_path).name
            outpath = GENERATED_DIR / obj_id / fname
            outpath.parent.mkdir(exist_ok=True, parents=True)
            if outpath.exists():
                continue
            url = f"{BASE_URL}{file_path}"
            res = client.get(url, headers=headers)
            res.raise_for_status()
            with open(outpath, 'wb') as f:
                f.write(res.content)



{'id': 'cbd2b8b22srmt0cwv53s83n08m', 'url': '/predictions/cbd2b8b22srmt0cwv53s83n08m', 'model': 'madhusanka-slc/super-slc-v1', 'version': 'db2ecb62be5357f4673bdb959aea07650843f500837645d79e13965be242660b', 'status': 'succeeded', 'created_at': '2026-03-10T19:44:19.990000Z', 'completed_at': '2026-03-10T19:44:36.069157Z', 'files': ['/predictions/cbd2b8b22srmt0cwv53s83n08m/files/0.jpg', '/predictions/cbd2b8b22srmt0cwv53s83n08m/files/1.jpg'], 'num_outputs': 2}
{'id': 'bqgkg0mhwnrmt0cwv53a0kk138', 'url': '/predictions/bqgkg0mhwnrmt0cwv53a0kk138', 'model': 'madhusanka-slc/super-slc-v1', 'version': 'db2ecb62be5357f4673bdb959aea07650843f500837645d79e13965be242660b', 'status': 'succeeded', 'created_at': '2026-03-10T19:43:26.693000Z', 'completed_at': '2026-03-10T19:43:43.122835Z', 'files': ['/predictions/bqgkg0mhwnrmt0cwv53a0kk138/files/0.jpg', '/predictions/bqgkg0mhwnrmt0cwv53a0kk138/files/1.jpg'], 'num_outputs': 2}
{'id': 't1fja6t90xrmt0cwv3jspg4fym', 'url': '/predictions/t1fja6t90xrmt0cwv3jspg